<a href="https://colab.research.google.com/github/g4utam262/ML_LAB/blob/main/EXP_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


In [16]:
# Load Titanic dataset
df = pd.read_csv("Titanic.csv")


In [17]:
# Select features
features = [
    "Pclass", "Sex", "Age", "SibSp",
    "Parch", "Fare", "Embarked"
]

X = df[features]
y = df["Survived"]


In [18]:
# Feature types
numeric_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare"
]

categorical_features = ["Sex", "Embarked"]


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

In [20]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)


In [21]:
# Base Decision Tree
base_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)


In [25]:
# Bagging
bagging = BaggingClassifier(
    estimator=base_model,
    n_estimators=50,
    random_state=42
)

bagging_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", bagging)
])

bagging_pipeline.fit(X_train, y_train)
bag_pred = bagging_pipeline.predict(X_test)

In [26]:
from sklearn.impute import SimpleImputer

# Redefine preprocessor to handle missing values (NaNs)
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_imputed = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numeric_features),
        ("cat", cat_transformer, categorical_features)
    ]
)

# AdaBoost with imputed pipeline
boosting = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=2,
        random_state=42
    ),
    n_estimators=50,
    random_state=42
)

boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor_imputed),
    ("classifier", boosting)
])

boosting_pipeline.fit(X_train, y_train)
boost_pred = boosting_pipeline.predict(X_test)

In [27]:
# Evaluate and compare models
def evaluate_predictions(y_true, y_pred, name):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred)
    }

# Compute metrics
bagging_metrics = evaluate_predictions(y_test, bag_pred, "Bagging Classifier")
boosting_metrics = evaluate_predictions(y_test, boost_pred, "AdaBoost Classifier")

# Display as DataFrame
metrics_df = pd.DataFrame([bagging_metrics, boosting_metrics])
print(metrics_df.to_string(index=False))

              Model  Accuracy  Precision   Recall  F1-Score
 Bagging Classifier  0.798883   0.866667 0.565217  0.684211
AdaBoost Classifier  0.776536   0.763636 0.608696  0.677419


In [28]:
# Evaluation function
def evaluate_model(name, y_true, y_pred):
    print("\n", name)

    print("Accuracy:",
          round(accuracy_score(y_true, y_pred), 4))

    print("Precision:",
          round(precision_score(y_true, y_pred), 4))

    print("Recall:",
          round(recall_score(y_true, y_pred), 4))

    print("F1-score:",
          round(f1_score(y_true, y_pred), 4))


In [29]:
# Evaluate
evaluate_model("Bagging", y_test, bag_pred)
evaluate_model("Boosting (AdaBoost)", y_test, boost_pred)



 Bagging
Accuracy: 0.7989
Precision: 0.8667
Recall: 0.5652
F1-score: 0.6842

 Boosting (AdaBoost)
Accuracy: 0.7765
Precision: 0.7636
Recall: 0.6087
F1-score: 0.6774
